# Standard Z7010 Gen 2 / OS 2 field test

Retained Z7010 reference workflow: this Pro development branch does not yet package a Z7020 image, and its loader rejects Z7020. Copy the template only to a new test.ipynb; preserve existing local results. This workflow uses the original fork BIN and matching Z7010 OS 2 overlay. Run cells step by step.

## 1. Select the board and image

Edit the hostname. This cell only checks which local PyRPL installation and fork bitstream will be used.

In [ ]:
import json
from pathlib import Path
import sys

import pyrpl
from pyrpl.redpitaya import RedPitaya

HOSTNAME = "rp-xxxxxx.local"
SSH_USER = "root"
bitstream = Path(pyrpl.__file__).resolve().parent / "fpga" / "red_pitaya.bin"

print("Python:", sys.executable)
print("PyRPL:", pyrpl.__file__)
print("Bitstream:", bitstream)
assert bitstream.is_file()

## 2. Run the read-only preflight

Enter the SSH password. This checks the OS, board profile, overlay contract, and local FPGA files without uploading or programming anything.

In [ ]:
from getpass import getpass

SSH_PASSWORD = getpass("SSH password: ")
device = None
try:
    device = RedPitaya(
        config=None,
        hostname=HOSTNAME,
        user=SSH_USER,
        password=SSH_PASSWORD,
        gui=False,
        autostart=False,
        reloadfpga=False,
        reloadserver=False,
    )
    preflight_report = device.preflight_fpga_update(filename=str(bitstream))
finally:
    if device is not None:
        device.end_ssh()

print(json.dumps(preflight_report, indent=2, sort_keys=True))

## 3. Program the FPGA

Uploads the fork BIN and matching DTBO and programs the FPGA, without starting the PyRPL server. This session targets the standard Z7010 Gen 2 board identified in step 2.

In [ ]:
device = None
try:
    device = RedPitaya(
        config=None,
        hostname=HOSTNAME,
        user=SSH_USER,
        password=SSH_PASSWORD,
        filename=str(bitstream),
        gui=False,
        autostart=False,
        reloadfpga=False,
        reloadserver=False,
    )
    program_report = device.update_fpga(filename=str(bitstream))
finally:
    if device is not None:
        device.end_ssh()

print(json.dumps(program_report, indent=2, sort_keys=True))

## 4. Start PyRPL and connect

Installs/starts the monitor server and checks register metadata, without reprogramming the FPGA. Saved PyRPL module settings are also applied; leave experimental actuators disconnected for these bench tests.

In [ ]:
from pyrpl import Pyrpl

p = Pyrpl(
    config="gen2-os2-field-test",
    hostname=HOSTNAME,
    user=SSH_USER,
    password=SSH_PASSWORD,
    filename=str(bitstream),
    gui=False,
    reloadfpga=False,
    reloadserver=True,
)
rp = p.rp
print("PyRPL connected; fork register checks passed.")

## 5. Read the fork's PID metadata

Reads hardware constants and sequence state. The author's RTL uses PSR=12, ISR=32, GAINBITS=30; this is a useful comparison, not a complete fingerprint or an analog test.

In [ ]:
pid = rp.pid0
print("PID0 input filter:", pid.inputfilter)
pid_metadata = {
    "PSR": int(pid._read(0x200)),
    "ISR": int(pid._read(0x204)),
    "GAINBITS": int(pid._read(0x20C)),
    "sequence_index": int(pid.setpoint_index),
    "sequence_setpoint": float(pid.setpoint_in_sequence),
}
print(json.dumps(pid_metadata, indent=2))

## 6. Prepare the bench workflow

This cell only defines local settings and a scope helper. Select IN/OUT 1 or 2 and record the actual load. Gen 2 full scale is ±2 V at Hi-Z or ±1 V at 50 Ω; PyRPL's existing numerical scale is unchanged. The helper configures/acquires the scope when called.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

TEST_CHANNEL = 1
OUTPUT_LOAD = "Hi-Z"  # Record the actual termination; this does not configure hardware.
BENCH_NOTES = ""  # Meter/scope, input jumper, cabling, measured amplitude/offset.
asg = rp.asg0
scope = rp.scope
physical_input = f"in{TEST_CHANNEL}"
physical_output = f"out{TEST_CHANNEL}"

def capture_scope(input1, input2, duration=0.02):
    scope.setup(input1=input1, input2=input2, duration=duration,
                trigger_source="immediately", trigger_delay=0,
                rolling_mode=False, average=False,
                ch1_active=True, ch2_active=True)
    try:
        data = np.asarray(scope.curve(timeout=5))
        times = scope.times.copy()
    finally:
        scope.stop()
    fig, ax = plt.subplots()
    ax.plot(times, data[0], label=input1)
    ax.plot(times, data[1], label=input2)
    ax.set(xlabel="Time [s]", ylabel="PyRPL signal units")
    ax.legend()
    plt.show()
    print("Channel means:", np.mean(data, axis=1))
    print("Channel peak-to-peak:", np.ptp(data, axis=1))
    return times, data

## 7. Disconnect fast-output routes

This changes routing for all PyRPL modules with direct fast outputs, isolating the upcoming bench signal. It does not program the FPGA. Outputs are not suitable for a connected experimental plant during these tests.

In [ ]:
for name in ("asg0", "asg1", "pid0", "pid1", "pid2",
             "iq0", "iq1", "iq2", "trig0", "trig1"):
    getattr(rp, name).output_direct = "off"
print("Fast-output routes disconnected.")

## 8. Generate and measure a small sine

Connect the selected OUT to the selected IN (LV) and an external instrument with the recorded termination. This enables a 1 kHz sine at amplitude 0.05 in PyRPL units. The output stays enabled until step 10; measure physical amplitude/offset before continuing.

In [ ]:
asg.setup(waveform="sin", frequency=1000, amplitude=0.05,
          offset=0, trigger_source="immediately", output_direct="off")
asg.output_direct = physical_output
print("Enabled", physical_output, "at 1 kHz; recorded load:", OUTPUT_LOAD)

## 9. Compare ADC and internal ASG signals

Configures/acquires the scope, then computes a spectrum locally. Compare the captured input with the external instrument; their ratio depends on termination and calibration, so it is not silently corrected.

In [ ]:
times, trace = capture_scope(physical_input, "asg0")
frequency = np.fft.rfftfreq(trace.shape[1], times[1] - times[0])
spectrum = np.abs(np.fft.rfft(trace[0] - np.mean(trace[0])))
peak = 1 + np.argmax(spectrum[1:])
print("Measured input frequency [Hz]:", frequency[peak])
fig, ax = plt.subplots()
ax.plot(frequency, spectrum)
ax.set(xlabel="Frequency [Hz]", ylabel="FFT magnitude (not calibrated PSD)",
       xlim=(0, 5000))
plt.show()

## 10. Disconnect the ASG output

Disables the physical ASG route. The following PID tests use internal signals only.

In [ ]:
asg.output_direct = "off"
print("ASG physical output disconnected.")

## 11. Exercise the proportional path

Configures PID0 with an internal ASG input and no physical output. Its trace should follow half the ASG signal. This does not tune or close an experimental feedback loop.

In [ ]:
pid.setup(input="asg0", output_direct="off", p=0.5, i=0,
          setpoint=0, inputfilter=0, min_voltage=-0.2, max_voltage=0.2,
          pause_gains="pi", paused=False, differential_mode_enabled=False)
pid.use_setpoint_sequence = False
pid.ival = 0
times, proportional_trace = capture_scope("asg0", "pid0")

## 12. Check hold and release

Pauses PID0 while its input sine continues, then releases it. During hold the PID trace should keep its last value, including the proportional contribution; after release it should follow the input again.

In [ ]:
pid.paused = True
try:
    times, held_trace = capture_scope("asg0", "pid0")
finally:
    pid.paused = False
times, released_trace = capture_scope("asg0", "pid0")

## 13. Exercise the integrator path

Temporarily sets I=100 Hz and P=0, with the existing ±0.2 internal output limits. Observe the phase/amplitude against the sine; the cell then clears the integrator and restores the proportional test.

In [ ]:
pid.p = 0
pid.ival = 0
pid.i = 100
try:
    times, integral_trace = capture_scope("asg0", "pid0")
finally:
    pid.i = 0
    pid.ival = 0
    pid.p = 0.5

## 14. Write and step all 16 setpoints

Writes a small signed sequence and advances it with software pulses, checking index, signed readback, and wrap. Keep DIO3_P low during this step. This checks the register path, not the physical TTL input.

In [ ]:
asg.setup(waveform="dc", amplitude=0, offset=0, output_direct="off",
          trigger_source="immediately")
sequence = np.linspace(-0.05, 0.05, 16)
pid.paused = True
pid.set_setpoint_array(sequence)
pid.reset_sequence_index()
pid.use_setpoint_sequence = True
pid.paused = False
sequence_results = []
for step in range(17):
    index = int(pid.setpoint_index)
    value = float(pid.setpoint_in_sequence)
    expected_index = step % 16
    expected_value = round(float(sequence[expected_index]) * 8192) / 8192
    row = {"step": step, "index": index, "setpoint": value,
           "expected": expected_value, "wrap": bool(pid.sequence_wrap_flag)}
    sequence_results.append(row)
    print(row)
    assert index == expected_index and value == expected_value, row
    assert row["wrap"] == (step == 16), row
    if step < 16:
        pid.manually_change_setpoint()

## 15. Check the external TTL inputs

Use a common ground and 3.3 V logic: DIO3_P rising edges advance PID0's sequence; DIO0_P high holds PID0. Apply pulses and rerun this cell to inspect state/trace. For hold, step the sequence while DIO0_P is high, then release it. All fast-output routes remain disconnected.

In [ ]:
print("Sequence index:", pid.setpoint_index,
      "setpoint:", pid.setpoint_in_sequence,
      "wrap:", pid.sequence_wrap_flag)
times, ttl_trace = capture_scope("asg0", "pid0", duration=0.1)

## 16. Inspect slow analog inputs

Enables automatic XADC conversion and reads the four inputs. Apply known voltages to the channels being used and compare with an external meter. Do not infer slow-output scaling from this input test.

In [ ]:
rp.ams.trigger_source = "auto"
print("Slow analog inputs (PyRPL units):", rp.ams.vadcs)

## 17. End the bench signal configuration

Stops acquisition, clears the tested PID's gains/integrator, disables sequence mode and ASG generation, and leaves fast-output routes disconnected. It does not restore earlier experiment settings.

In [ ]:
scope.stop()
pid.use_setpoint_sequence = False
pid.p = 0
pid.i = 0
pid.ival = 0
pid.setpoint = 0
pid.paused = False
asg.amplitude = 0
asg.offset = 0
asg.trigger_source = "off"
for name in ("asg0", "asg1", "pid0", "pid1", "pid2",
             "iq0", "iq1", "iq2", "trig0", "trig1"):
    getattr(rp, name).output_direct = "off"

## 18. Reconnect without reprogramming

Closes this PyRPL instance (including its server), then starts/connects again with both reload requests disabled. This still starts a monitor server and applies module settings; it is not read-only.

In [ ]:
p._clear()
p = Pyrpl(config="gen2-os2-field-test", hostname=HOSTNAME,
          user=SSH_USER, password=SSH_PASSWORD,
          filename=str(bitstream), gui=False,
          reloadfpga=False, reloadserver=False)
rp = p.rp
print("Reconnected; PID0 input filter:", rp.pid0.inputfilter)

## Remaining measurements

Record physical amplitude/offset, termination, input jumper, instrument, test duration and observations in this local notebook. Closed-loop performance, high-speed TTL timing, slow outputs, and long-duration/GUI operation still need their own bench measurements. The internal traces and FFT above do not establish those results.